In [6]:
2+2

4

In [7]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re
import shutil


In [8]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [9]:
result_from_prev_analysis = pd.read_parquet("/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/results/results_myvideos_174_rec_session_3/logits_output_meta.parquet")
print(result_from_prev_analysis.shape)
del result_from_prev_analysis['top5_classes']
result_from_prev_analysis.head(4)

(99, 13)


,video_id,cam,session,template,placeholders,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,261008-111314,cam0,mime_pretend_poke,Pretending to poke [something],nothing,69,69,Pretending to poke [something],0.97,"[69, 81, 128, 124, 78]",True,True
1,261008-111314,cam1,mime_pretend_poke,Pretending to poke [something],nothing,69,69,Pretending to poke [something],0.89,"[69, 81, 124, 78, 134]",True,True
2,261008-111314,cam2,mime_pretend_poke,Pretending to poke [something],nothing,69,80,Pretending to squeeze [something],0.27,"[80, 69, 74, 128, 122]",False,True
3,261008-111608,cam0,mime_pretend_poke,Pretending to poke [something],nothing,69,69,Pretending to poke [something],0.60,"[69, 128, 81, 68, 124]",True,True


In [10]:
result = pd.read_parquet("./predictions.parquet")
print(result.shape)
result.sample(2)

(99, 4)


,video_id,label,top1_label,correct_top1
83,261008-114837_cam2,36,80,False
65,261008-114331_cam2,2,159,False


In [11]:
# rename top1_label to top1_pred
result = result.rename(columns={'top1_label':'top1_pred'})

# I want to break video_id into video_id and cam
result[['video_id', 'cam']] = result.video_id.str.split('_', expand=True)
print(result.shape)

# Bring in session column with a left join
# Bring in template column with a left join (to later verify match rate with my custom template2id mapping.)
result_from_prev_analysis = result_from_prev_analysis[['video_id','cam', 'session','template']]
result = pd.merge(result, result_from_prev_analysis, how = 'left', on = ['video_id','cam']) # LEFT JOIN
print(result.shape)

result.sample(3)

(99, 5)
(99, 7)


,video_id,label,top1_pred,correct_top1,cam,session,template
80,261008-114453,2,65,False,cam2,bending_1,Bending [something] so that it deforms
53,261008-113415,68,65,False,cam2,mime_pretend_pickUp,Pretending to pick [something] up
71,261008-114350,2,93,False,cam2,mime_bending,Bending [something] so that it deforms


In [12]:
d = np.load("./predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(99, 174)


NpzFile './predictions.npz' with keys: logits, probs, video_ids, labels

# main result

In [14]:

n = len(result)
top1_acc  = result["correct_top1"].mean()

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 4),
}
my_metrics

{'total_evaluated': 99, 'top1_accuracy': np.float64(0.1414)}

- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [15]:
result.groupby("cam")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

,top1_acc
cam,
cam0,0.15
cam1,0.24
cam2,0.03


In [16]:
result.groupby("template")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
).sort_values("template")

,top1_acc
template,
Bending [something] so that it deforms,0.00
Moving [something] and [something] closer to each other,0.33
Pretending to pick [something] up,0.22
Pretending to poke [something],0.07
Pretending to sprinkle air onto [something],0.11


In [17]:
result[result.cam=='cam1'].groupby("template")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
).sort_values("template")

,top1_acc
template,
Bending [something] so that it deforms,0.00
Moving [something] and [something] closer to each other,0.50
Pretending to pick [something] up,0.33
Pretending to poke [something],0.11
Pretending to sprinkle air onto [something],0.33


In [37]:
result[result.cam=='cam1'].groupby(["template","session"])[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

top1_acc
template                                                session                             
Bending [something] so that it deforms                  bending_1                    0.00000
                                                        mime_bending                 0.00000
Moving [something] and [something] closer to each other mime_moveCloserToEachOther   0.00000
                                                        moveCloserToEachOther_1      1.00000
Pretending to pick [something] up                       mime_pretend_pickUp          0.00000
                                                        pretend_pickUp_1             0.66667
Pretending to poke [something]                          mime_pretend_poke            0.16667
                                                        pretend_poke_1               0.00000
Pretending to sprinkle air onto [something]             mime_pretend_sprinkle        0.33333
                                                        pretend_sprinkle_1           0.33333

# Granular analysis

##### Distractor objects -> expected spike in action/verb -- P(Action|Noun) estimate
- Plug -> Plugging [something] into [something] -- 55%
- lady/person -> Showing a photo of [something] to the camera -- 96%
- magnet -> Attaching [something] to [something] -- 60%
- Books -> Piling [something] up -- 55%
- (~) Paper -> Tearing [something] into two pieces -- 25%

##### id2label

In [20]:
# Load mapping text file into a Series with template strings as the index
template2id_mapping = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/experiment_ssv2_object_occlusion/data/template2id_mapping.txt', sep='\t', header=None, index_col=1)[0]
template2id_mapping

1
Approaching [something] with your camera                                 0
Attaching [something] to [something]                                     1
Bending [something] so that it deforms                                   2
Bending [something] until it breaks                                      3
Burying [something] in [something]                                       4
                                                                      ... 
[Something] being deflected from [something]                           169
[Something] colliding with [something] and both are being deflected    170
[Something] colliding with [something] and both come to a halt         171
[Something] falling like a feather or paper                            172
[Something] falling like a rock                                        173
Name: 0, Length: 174, dtype: int64

In [21]:
# Open text file into a dict with label string as key and integer ID as value
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/experiment_ssv2_object_occlusion/data/template2id_mapping.txt') as f:
    id2label_dict = {int(line.strip().split('\t')[0]) : line.strip().split('\t')[1] for line in f}

id2label_dict

{0: 'Approaching [something] with your camera',
 1: 'Attaching [something] to [something]',
 2: 'Bending [something] so that it deforms',
 3: 'Bending [something] until it breaks',
 4: 'Burying [something] in [something]',
 5: 'Closing [something]',
 6: 'Covering [something] with [something]',
 7: 'Digging [something] out of [something]',
 8: 'Dropping [something] behind [something]',
 9: 'Dropping [something] in front of [something]',
 10: 'Dropping [something] into [something]',
 11: 'Dropping [something] next to [something]',
 12: 'Dropping [something] onto [something]',
 13: 'Failing to put [something] into [something] because [something] does not fit',
 14: 'Folding [something]',
 15: 'Hitting [something] with [something]',
 16: 'Holding [something]',
 17: 'Holding [something] behind [something]',
 18: 'Holding [something] in front of [something]',
 19: 'Holding [something] next to [something]',
 20: 'Holding [something] over [something]',
 21: 'Laying [something] on the table on 

In [22]:
# Apply mapping to create new column
result['template_custom'] = result['label'].map(id2label_dict)
result.sample(7)

,video_id,label,top1_pred,correct_top1,cam,session,template,template_custom
58,261008-113443,68,119,False,cam1,pretend_pickUp_1,Pretending to pick [something] up,Pretending to pick [something] up
46,261008-113356,68,111,False,cam1,mime_pretend_pickUp,Pretending to pick [something] up,Pretending to pick [something] up
67,261008-114344,2,80,False,cam1,mime_bending,Bending [something] so that it deforms,Bending [something] so that it deforms
80,261008-114453,2,65,False,cam2,bending_1,Bending [something] so that it deforms,Bending [something] so that it deforms
13,261008-111649,69,93,False,cam1,mime_pretend_poke,Pretending to poke [something],Pretending to poke [something]
56,261008-113428,68,68,True,cam2,pretend_pickUp_1,Pretending to pick [something] up,Pretending to pick [something] up
82,261008-114837,36,35,False,cam1,mime_moveCloserToEachOther,Moving [something] and [something] closer to each other,Moving [something] and [something] closer to each other


In [23]:
pd.testing.assert_series_equal(result.template, result.template_custom, check_names=False, check_exact=False)

In [24]:
np.mean(result.template == result.template_custom)*100

np.float64(100.0)

In [25]:
label2id_dict = {v: k for k, v in id2label_dict.items()}

print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[(0, 'Approaching [something] with your camera'), (1, 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', 0), ('Attaching [something] to [something]', 1)]


In [26]:
result.head(1)

,video_id,label,top1_pred,correct_top1,cam,session,template,template_custom
0,261008-111314,69,93,False,cam0,mime_pretend_poke,Pretending to poke [something],Pretending to poke [something]


In [27]:
df = result[['video_id',	'cam'	,'session'	,'template',	'label'	,'top1_pred','correct_top1']]
print(df.shape)
df.head(2)

(99, 7)


,video_id,cam,session,template,label,top1_pred,correct_top1
0,261008-111314,cam0,mime_pretend_poke,Pretending to poke [something],69,93,False
1,261008-111314,cam1,mime_pretend_poke,Pretending to poke [something],69,80,False


In [28]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,template,label,top1_pred,correct_top1,prob_0,prob_1,prob_2,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,261008-111314,cam0,mime_pretend_poke,Pretending to poke [something],69,93,False,0.003072,0.001844,0.004001,...,0.000808,0.010048,0.000831,0.000818,0.015891,0.000539,0.007109,0.002556,0.000439,0.001059
1,261008-111314,cam1,mime_pretend_poke,Pretending to poke [something],69,80,False,0.001081,0.001564,0.006544,...,0.005832,0.052991,0.001045,0.000738,0.015815,0.000456,0.001792,0.000831,0.000290,0.001295


#### Main Analysis 


In [29]:
pd.set_option('display.float_format',  '{:,.5f}'.format)

In [34]:
def avg_prob_in_diff_cases(df: pd.DataFrame, session_type: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[(df['cam'].isin(cams)) & (df['session'].str.contains(session_type))].copy()
    
    r = df_temp.groupby('session').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        # top5acc=('correct_top5', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session').reset_index()

    print("=" * 25, session_type, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos', "=" * 20)

    dist_df = df_temp

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/apply_dinov2_attn_probe/data/myvideos_3/frames/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = (dist_df['video_id'] + '_' + dist_df['cam']).dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

# Analysis special



In [35]:
df.session.unique()

array(['mime_pretend_poke', 'pretend_poke_1', 'mime_pretend_sprinkle',
       'pretend_sprinkle_1', 'mime_pretend_pickUp', 'pretend_pickUp_1',
       'mime_bending', 'bending_1', 'mime_moveCloserToEachOther',
       'moveCloserToEachOther_1'], dtype=object)

In [36]:
avg_prob_in_diff_cases(df, session_type='bending', label='Bending [something] so that it deforms', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= bending =========================
label 2: Bending [something] so that it deforms
cams used: ['cam1']
        session  videos  prob_mean  prob_std  top1acc                                     examples
0     bending_1       3    0.09354   0.05695  0.00000  261008-114415, 261008-114428, 261008-114453
1  mime_bending       3    0.02317   0.01849  0.00000  261008-114331, 261008-114344, 261008-114350
==================== Videos ====================
         video_id   cam       session                                template label top1_pred correct_top1 prob_65 prob_64 prob_165 prob_80 prob_157
64  261008-114331  cam1  mime_bending  Bending [something] so that it deforms     2        65        False 0.25810 0.14973  0.09690 0.09298  0.09049

Top 5 Probability Breakdown:
  prob_65  ->  0.25810  |  Pretending to be tearing [something that is not tearable]
  prob_64  ->  0.14973  |  Pretending or trying and failing to twist [something]
  prob_165 ->  0.09690  |  Twist

In [38]:
avg_prob_in_diff_cases(df, session_type='Closer', label='Moving [something] and [something] closer to each other', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= Closer =========================
label 36: Moving [something] and [something] closer to each other
cams used: ['cam1']
                      session  videos  prob_mean  prob_std  top1acc                                     examples
0  mime_moveCloserToEachOther       3    0.16366   0.02659  0.00000  261008-114837, 261008-114847, 261008-114901
1     moveCloserToEachOther_1       3    0.46465   0.28662  1.00000  261008-114921, 261008-114942, 261008-114959
==================== Videos ====================
         video_id   cam                     session                                                 template label top1_pred correct_top1 prob_35 prob_36 prob_144 prob_65 prob_170
82  261008-114837  cam1  mime_moveCloserToEachOther  Moving [something] and [something] closer to each other    36        35        False 0.24399 0.18898  0.11026 0.05341  0.04749

Top 5 Probability Breakdown:
  prob_35  ->  0.24399  |  Moving [something] and [something] away from each 

In [39]:
avg_prob_in_diff_cases(df, session_type='pick', label='Pretending to pick [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= pick =========================
label 68: Pretending to pick [something] up
cams used: ['cam1']
               session  videos  prob_mean  prob_std  top1acc                                     examples
0  mime_pretend_pickUp       3    0.05336   0.01158  0.00000  261008-113356, 261008-113406, 261008-113415
1     pretend_pickUp_1       3    0.13860   0.08690  0.66667  261008-113428, 261008-113443, 261008-113455
==================== Videos ====================
         video_id   cam              session                           template label top1_pred correct_top1 prob_111 prob_74 prob_44 prob_21 prob_47
46  261008-113356  cam1  mime_pretend_pickUp  Pretending to pick [something] up    68       111        False  0.14331 0.11035 0.09492 0.08101 0.06845

Top 5 Probability Breakdown:
  prob_111 ->  0.14331  |  Putting [something] on a surface
  prob_74  ->  0.11035  |  Pretending to put [something] on a surface
  prob_44  ->  0.09492  |  Moving [something] up
  p

In [40]:
avg_prob_in_diff_cases(df, session_type='poke', label='Pretending to poke [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= poke =========================
label 69: Pretending to poke [something]
cams used: ['cam1']
             session  videos  prob_mean  prob_std  top1acc                                     examples
0  mime_pretend_poke       6    0.04486   0.08755  0.16667  261008-111314, 261008-111608, 261008-111617
1     pretend_poke_1       3    0.14218   0.02892  0.00000  261008-112131, 261008-112156, 261008-112220
==================== Videos ====================
        video_id   cam            session                        template label top1_pred correct_top1 prob_80 prob_138 prob_165 prob_64 prob_84
1  261008-111314  cam1  mime_pretend_poke  Pretending to poke [something]    69        80        False 0.19356  0.16155  0.05299 0.03546 0.03402

Top 5 Probability Breakdown:
  prob_80  ->  0.19356  |  Pretending to squeeze [something]
  prob_138 ->  0.16155  |  Squeezing [something]
  prob_165 ->  0.05299  |  Twisting [something]
  prob_64  ->  0.03546  |  Pretending or tr

In [41]:
avg_prob_in_diff_cases(df, session_type='sprinkle', label='Pretending to sprinkle air onto [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= sprinkle =========================
label 79: Pretending to sprinkle air onto [something]
cams used: ['cam1']
                 session  videos  prob_mean  prob_std  top1acc                                     examples
0  mime_pretend_sprinkle       3    0.06283   0.06162  0.33333  261008-112813, 261008-112823, 261008-112834
1     pretend_sprinkle_1       3    0.13731   0.08417  0.33333  261008-112944, 261008-113003, 261008-113021
==================== Videos ====================
         video_id   cam                session                                     template label top1_pred correct_top1 prob_79 prob_44 prob_84 prob_25 prob_137
28  261008-112813  cam1  mime_pretend_sprinkle  Pretending to sprinkle air onto [something]    79        79         True 0.13351 0.10996 0.04408 0.03972  0.03734

Top 5 Probability Breakdown:
  prob_79  ->  0.13351  |  Pretending to sprinkle air onto [something]
  prob_44  ->  0.10996  |  Moving [something] up
  prob_84  ->  0.0